In [9]:
import os
from dotenv import load_dotenv

# Load secret API key from .env file (kept safe and not uploaded to GitHub)
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = "your_api_key_here"


In [10]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv
!pip install -q youtube-transcript-api langchain-community langchain-google-genai langchain-text-splitters faiss-cpu tiktoken python-dotenv
# !pip install -q sentence-transformers

In [11]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings

In [12]:
import os
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id = "QXcYY4PyDs4"
cache_file = "transcript.txt"

if os.path.exists(cache_file):
    print(f"Loading transcript from local cache ({cache_file})...")
    with open(cache_file, "r", encoding="utf-8") as f:
        transcript = f.read()
    print(f"Successfully loaded transcript ({len(transcript)} chars).")
else:
    try:
        ytt = YouTubeTranscriptApi()
        transcript_list = ytt.fetch(video_id, languages=["en"])
        transcript = " ".join(chunk.text for chunk in transcript_list)
        with open(cache_file, "w", encoding="utf-8") as f:
            f.write(transcript)
        print(f"Successfully fetched and saved transcript ({len(transcript)} chars).")
    except TranscriptsDisabled:
        print("No captions available for this video.")


Loading transcript from local cache (transcript.txt)...
Successfully loaded transcript (121384 chars).


In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [14]:
len(chunks)

53

In [15]:
chunks[0]

Document(metadata={}, page_content='This would begin when the life of all the universes was being debated in the tournament of power. [music] The tournament of power had reached a very key moment where Goku would face the warrior Jiren of universe 11. Both were fighting but the superiority of this guy before the Saiyan was more [music] than evident. As initially had happened, Goku did not occur to him better idea than to make the [music] Genk Kai Dama. This for him could mean his elimination. if he got to kill Jiren for that reason. [music] But this did not even pass through the mind of the Saiyan. Anyway, Goku [music] would ask for the help of all the others to create the Genkid Dama with the help of everyone [music] present. Everyone wanted to get rid of Jiren, so they would give their energies. The Genkid [music] Dama would be created to later throw it to Jiren. But surprisingly, Jiren was giving it back to Goku. Things were complicated since Goku despite using all his power [music]

In [16]:
print(f"Embedding {len(chunks)} chunks with Gemini... (takes ~15-20 seconds)")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = FAISS.from_documents(chunks, embeddings)
print(" Vector store successfully built!")

Embedding 53 chunks with Gemini... (takes ~15-20 seconds)


 Vector store successfully built!


In [17]:
vector_store.index_to_docstore_id

{0: '70b412d6-f1a9-43f0-8cbb-b862d4dca226',
 1: '628b3720-e9a1-4ccd-8ca7-a6dc3239800b',
 2: 'fa9563a8-7982-4e32-aae2-7cc35a533260',
 3: 'c09b86fc-5a50-4a90-839a-2ca07e12727f',
 4: '2a6db5f7-e8a6-4523-9353-05b46c197f6b',
 5: 'fdb01615-f844-4070-8edd-d4e588b9a049',
 6: '084b2b8d-1be7-4a03-8e23-120e964e17f7',
 7: '84f00070-5cb2-4167-aaa0-890492eaf415',
 8: 'e909b3cd-6b23-459e-b0f0-56bd636e5d90',
 9: 'ac11c2fc-cfc1-4c0d-b7fa-c2b949ea2ecc',
 10: '0f31019d-4837-4fdf-88dd-32c2d53c1459',
 11: '5a93a949-ccdf-4796-ba29-579957699077',
 12: '8359af6b-cd74-43a0-8caf-e4e07851cf7d',
 13: 'd66784bf-f2e2-46e5-ba45-ad24bfed962e',
 14: '07ab2a7b-f04c-4713-bead-13458c9686ea',
 15: 'e279649f-5281-433e-ad65-17da7e5c52f8',
 16: 'b5ee00ab-b2c7-48a6-a337-017572966969',
 17: 'c9625d64-119a-434f-a032-e9a25085a231',
 18: '9488fd2d-e9a2-41ea-a4ac-6ae7963bb2e3',
 19: '0aaf7bcc-15cf-45de-93fd-fee713125add',
 20: '0c3b59d0-d34c-4abb-80e5-66ac51416e91',
 21: '4a28bd11-716d-45ac-acd9-d3460452200f',
 22: 'e56995c3-bc52-

In [18]:
vector_store.get_by_ids(['5038dfcc-1eda-42d4-a684-8160020aafd0'])

[]

In [19]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever


VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7703d76ed400>, search_kwargs={'k': 4})

In [28]:
query = "What happened to Goku?"
retriever.invoke(query)
# print("Top retrieved chunk:\n", docs[0].page_content)   

[Document(id='70b412d6-f1a9-43f0-8cbb-b862d4dca226', metadata={}, page_content='This would begin when the life of all the universes was being debated in the tournament of power. [music] The tournament of power had reached a very key moment where Goku would face the warrior Jiren of universe 11. Both were fighting but the superiority of this guy before the Saiyan was more [music] than evident. As initially had happened, Goku did not occur to him better idea than to make the [music] Genk Kai Dama. This for him could mean his elimination. if he got to kill Jiren for that reason. [music] But this did not even pass through the mind of the Saiyan. Anyway, Goku [music] would ask for the help of all the others to create the Genkid Dama with the help of everyone [music] present. Everyone wanted to get rid of Jiren, so they would give their energies. The Genkid [music] Dama would be created to later throw it to Jiren. But surprisingly, Jiren was giving it back to Goku. Things were complicated si

In [21]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.2)
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.2', 'langchain': '1.5.0', 'langchain-google-genai': '4.5.0'}}, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', temperature=0.2, client=<google.genai.client.Client object at 0x7703750a0410>, default_metadata=(), model_kwargs={}

In [22]:
prompt = PromptTemplate(
    template="""You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.
Provide your answer in clear, flowing sentences in paragraph form. Do not use bullet points, lists, or asterisks (*).

Context:
{context}

Question: {question}
Answer:""",
    input_variables=["context", "question"]
)


In [23]:
question          = "where is goku"
retrieved_docs    = retriever.invoke(question)

In [24]:
retrieved_docs

[Document(id='70b412d6-f1a9-43f0-8cbb-b862d4dca226', metadata={}, page_content='This would begin when the life of all the universes was being debated in the tournament of power. [music] The tournament of power had reached a very key moment where Goku would face the warrior Jiren of universe 11. Both were fighting but the superiority of this guy before the Saiyan was more [music] than evident. As initially had happened, Goku did not occur to him better idea than to make the [music] Genk Kai Dama. This for him could mean his elimination. if he got to kill Jiren for that reason. [music] But this did not even pass through the mind of the Saiyan. Anyway, Goku [music] would ask for the help of all the others to create the Genkid Dama with the help of everyone [music] present. Everyone wanted to get rid of Jiren, so they would give their energies. The Genkid [music] Dama would be created to later throw it to Jiren. But surprisingly, Jiren was giving it back to Goku. Things were complicated si

In [25]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'This would begin when the life of all the universes was being debated in the tournament of power. [music] The tournament of power had reached a very key moment where Goku would face the warrior Jiren of universe 11. Both were fighting but the superiority of this guy before the Saiyan was more [music] than evident. As initially had happened, Goku did not occur to him better idea than to make the [music] Genk Kai Dama. This for him could mean his elimination. if he got to kill Jiren for that reason. [music] But this did not even pass through the mind of the Saiyan. Anyway, Goku [music] would ask for the help of all the others to create the Genkid Dama with the help of everyone [music] present. Everyone wanted to get rid of Jiren, so they would give their energies. The Genkid [music] Dama would be created to later throw it to Jiren. But surprisingly, Jiren was giving it back to Goku. Things were complicated since Goku despite using all his power [music] could not be able to stop it. Fina

In [26]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don\'t know.\n\n      This would begin when the life of all the universes was being debated in the tournament of power. [music] The tournament of power had reached a very key moment where Goku would face the warrior Jiren of universe 11. Both were fighting but the superiority of this guy before the Saiyan was more [music] than evident. As initially had happened, Goku did not occur to him better idea than to make the [music] Genk Kai Dama. This for him could mean his elimination. if he got to kill Jiren for that reason. [music] But this did not even pass through the mind of the Saiyan. Anyway, Goku [music] would ask for the help of all the others to create the Genkid Dama with the help of everyone [music] present. Everyone wanted to get rid of Jiren, so they would give their energies. The Genkid [music] Dama would be cr

In [30]:
answer = llm.invoke(final_prompt)
print(answer.text)


Based on the provided transcript, Goku is on Planet Vegeta.


In [31]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [32]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [33]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [34]:
parallel_chain.invoke('what is vegita')

{'context': 'This would begin when the life of all the universes was being debated in the tournament of power. [music] The tournament of power had reached a very key moment where Goku would face the warrior Jiren of universe 11. Both were fighting but the superiority of this guy before the Saiyan was more [music] than evident. As initially had happened, Goku did not occur to him better idea than to make the [music] Genk Kai Dama. This for him could mean his elimination. if he got to kill Jiren for that reason. [music] But this did not even pass through the mind of the Saiyan. Anyway, Goku [music] would ask for the help of all the others to create the Genkid Dama with the help of everyone [music] present. Everyone wanted to get rid of Jiren, so they would give their energies. The Genkid [music] Dama would be created to later throw it to Jiren. But surprisingly, Jiren was giving it back to Goku. Things were complicated since Goku despite using all his power [music] could not be able to s

In [35]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | llm | parser

In [37]:
response = main_chain.invoke("Can you summarize the video")

# Remove any asterisks and print as clean sentences
clean_ans = response.replace("*", "").strip()
print(clean_ans)


Based on the provided transcript, the story involves several key events:

   Intervention at Planet Vegeta: Whis intervenes to stop a giant ball of energy from destroying Planet Vegeta and the Saiyans, an act that Beerus considers a violation of the rules. Whis knocks Beerus unconscious and takes him to Zeno Sama, who has guards wake Beerus up using explosives.
   The Tournament of Power: During the Tournament of Power, Goku uses a Genki Dama against Jiren. The attack fails, and Goku is consumed by the energy, leading to his disappearance. Universe 7 is erased by Zeno Sama, and Universe 11 wins.
   Goku's New Path: Goku finds himself on a ship in the past, witnessing a conversation between Bardock and Gine regarding Kakarot. Goku later meets Broly and recruits him to become the new God of Destruction for Universe 7, taking him to meet Zeno Sama and Daishin Khan.
   Conflict with Beerus: In a separate sequence, Goku and Beerus engage in a fight. Goku struggles because his body cannot ha